# VietEmbed-RAG V1 — VN-MTEB RAG Core Benchmark

Notebook dành riêng cho **VietEmbed-RAG V1**, model fine-tune cho retrieval/RAG.

Mục tiêu:
- Chỉ benchmark Retrieval
- Không chạy các corpus khổng lồ như MSMARCO, HotpotQA, FEVER, NQ
- Chạy thực tế trên Google Colab
- Dễ debug
- Chạy từng task một
- Có cache để chạy tiếp nếu Colab bị ngắt
- Dùng FP16 trên GPU

## Bộ benchmark mặc định

| Task | Corpus | Ý nghĩa |
|---|---:|---|
| SciFact-VN | ~5K | Scientific retrieval |
| NFCorpus-VN | ~10K | Medical retrieval |
| CQADupstackAndroid-VN | ~25K | Technical / Web QA |
| SCIDOCS-VN | ~38K | Academic retrieval |
| FiQA2018-VN | ~59K | Finance QA |
| TRECCOVID-VN | ~229K | Medical / Academic retrieval |
| Quora-VN | ~534K | General Web QA |

6 task lõi: khoảng 365K documents.  
Bật Quora: khoảng 900K documents.

## 1. Cài thư viện

In [ ]:
!pip install -q -U "mteb[xet]>=2.2.0" sentence-transformers

## 2. Kiểm tra môi trường

In [ ]:
import torch
import mteb
import sentence_transformers

print("PyTorch:", torch.__version__)
print("MTEB:", mteb.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Cấu hình

Thường chỉ cần sửa `MODEL_ZIP`.

- `INCLUDE_QUORA = True`: benchmark đầy đủ hơn cho general RAG
- `INCLUDE_QUORA = False`: test nhanh

In [ ]:
from pathlib import Path

if Path("/content").is_dir():
    RUNTIME_DIR = Path("/content")
elif Path("/kaggle/working").is_dir():
    RUNTIME_DIR = Path("/kaggle/working")
else:
    RUNTIME_DIR = Path.cwd()

MODEL_ZIP = "benchmark/VietEmbed-RAG.zip"
EXTRACT_DIR = RUNTIME_DIR / "VietEmbed-RAG-V1"

USE_E5_PREFIX = True
BATCH_SIZE = 32

INCLUDE_QUORA = True

RESULT_DIR = str(RUNTIME_DIR / "vietembed_v1_mteb_results")

## 4. Giải nén model

In [ ]:
from pathlib import Path
import shutil
import tempfile
import zipfile

def resolve_model_source(value):
    raw_path = Path(value).expanduser()
    candidates = [
        raw_path,
        Path.cwd() / raw_path,
        Path.cwd() / raw_path.name,
        Path.cwd().parent / raw_path,
        Path("/content") / raw_path.name,
    ]
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if candidate.exists():
            return candidate

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        matches = list(kaggle_input.rglob(raw_path.name))
        if len(matches) == 1:
            return matches[0].resolve()
        if len(matches) > 1:
            raise FileExistsError(
                f"Tìm thấy nhiều file tên {raw_path.name} trong /kaggle/input: {matches}"
            )

    checked = "\n".join(f"- {path}" for path in seen)
    raise FileNotFoundError(f"Không tìm thấy model. Đã kiểm tra:\n{checked}")


model_source = resolve_model_source(MODEL_ZIP)
extract_dir = Path(EXTRACT_DIR).resolve()

if model_source.is_dir():
    search_root = model_source
else:
    with model_source.open("rb") as stream:
        header = stream.read(512).lstrip()
    if header.startswith(b"version https://git-lfs.github.com/spec/v1"):
        raise RuntimeError(
            f"{model_source} chỉ là Git LFS pointer, chưa phải model ZIP thật."
        )
    if header.lower().startswith((b"<!doctype html", b"<html")):
        raise RuntimeError(
            f"{model_source} là trang HTML tải nhầm, không phải model ZIP."
        )
    if not zipfile.is_zipfile(model_source):
        raise zipfile.BadZipFile(
            f"File không phải ZIP hoàn chỉnh: {model_source} "
            f"({model_source.stat().st_size:,} bytes). Hãy chờ copy/upload xong rồi chạy lại."
        )

    extract_dir.parent.mkdir(parents=True, exist_ok=True)
    temporary_dir = Path(
        tempfile.mkdtemp(prefix=f".{extract_dir.name}.", dir=extract_dir.parent)
    )
    try:
        with zipfile.ZipFile(model_source) as archive:
            bad_member = archive.testzip()
            if bad_member is not None:
                raise zipfile.BadZipFile(f"CRC lỗi tại entry: {bad_member}")

            root = temporary_dir.resolve()
            for member in archive.infolist():
                destination = (temporary_dir / member.filename).resolve()
                if destination != root and root not in destination.parents:
                    raise zipfile.BadZipFile(
                        f"ZIP chứa đường dẫn không an toàn: {member.filename}"
                    )
            archive.extractall(temporary_dir)

        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        temporary_dir.replace(extract_dir)
    except Exception:
        shutil.rmtree(temporary_dir, ignore_errors=True)
        raise
    search_root = extract_dir

model_files = list(search_root.rglob("modules.json"))

valid_model_dirs = [
    path.parent
    for path in model_files
    if (path.parent / "config.json").is_file()
    and any(
        (path.parent / weight_name).is_file()
        for weight_name in ("model.safetensors", "pytorch_model.bin")
    )
]
if len(valid_model_dirs) != 1:
    raise RuntimeError(
        f"Cần đúng 1 SentenceTransformer model, tìm thấy {len(valid_model_dirs)} "
        f"trong {search_root}."
    )

MODEL_DIR = str(valid_model_dirs[0])

print("Model source:", model_source)
if model_source.is_file():
    print(f"ZIP size: {model_source.stat().st_size / 1024**2:.1f} MiB")
print("Model directory:", MODEL_DIR)

## 5. Load model

Với E5 retrieval:
- Query: `query: ...`
- Document: `passage: ...`

Nếu có GPU, model chuyển sang FP16.

In [ ]:
from sentence_transformers import SentenceTransformer

prompts = {}

if USE_E5_PREFIX:
    prompts = {
        "query": "query: ",
        "document": "passage: ",
    }

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    MODEL_DIR,
    device=device,
    prompts=prompts,
)

if device == "cuda":
    model.half()

model.eval()

print("Device:", device)
print("Embedding dimension:", model.get_embedding_dimension())
print("Max sequence length:", model.max_seq_length)
print("Prompts:", prompts)

## 6. Sanity check

In [ ]:
from sentence_transformers.util import cos_sim

if USE_E5_PREFIX:
    texts = [
        "query: Trí tuệ nhân tạo là gì?",
        "passage: Trí tuệ nhân tạo là lĩnh vực nghiên cứu các hệ thống có khả năng thực hiện những nhiệm vụ cần trí thông minh.",
        "passage: Hôm nay thời tiết tại thành phố có mưa lớn.",
    ]
else:
    texts = [
        "Trí tuệ nhân tạo là gì?",
        "Trí tuệ nhân tạo là lĩnh vực nghiên cứu các hệ thống có khả năng thực hiện những nhiệm vụ cần trí thông minh.",
        "Hôm nay thời tiết tại thành phố có mưa lớn.",
    ]

embeddings = model.encode(texts)

print("Embedding shape:", embeddings.shape)
print("Relevant:", cos_sim(embeddings[0], embeddings[1]).item())
print("Unrelated:", cos_sim(embeddings[0], embeddings[2]).item())

## 7. Chọn Retrieval tasks

In [ ]:
TASK_NAMES = [
    "SciFact-VN",
    "NFCorpus-VN",
    "CQADupstackAndroid-VN",
    "SCIDOCS-VN",
    "FiQA2018-VN",
    "TRECCOVID-VN",
]

if INCLUDE_QUORA:
    TASK_NAMES.append("Quora-VN")

tasks = mteb.get_tasks(tasks=TASK_NAMES)

print("Tasks:")
for name in TASK_NAMES:
    print("-", name)

## 8. Result Cache

MTEB lưu kết quả từng task. Nếu chạy lại, `only-missing` sẽ bỏ qua phần đã có.

In [ ]:
from pathlib import Path

Path(RESULT_DIR).mkdir(parents=True, exist_ok=True)

cache = mteb.ResultCache(cache_path=RESULT_DIR)

print("Cache:", RESULT_DIR)

# 9. Chạy benchmark

Task được sắp từ nhỏ đến lớn để dễ debug và có kết quả sớm.

Metric chính: **NDCG@10**.

In [ ]:
import time

total_start = time.perf_counter()

for i, task in enumerate(tasks, start=1):
    task_name = task.metadata.name

    print()
    print("=" * 70)
    print(f"[{i}/{len(tasks)}] {task_name}")
    print("=" * 70)

    start = time.perf_counter()

    mteb.evaluate(
        model,
        tasks=[task],
        cache=cache,
        encode_kwargs={"batch_size": BATCH_SIZE},
        overwrite_strategy="only-missing",
    )

    elapsed = (time.perf_counter() - start) / 60
    print(f"{task_name} finished in {elapsed:.1f} minutes")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

total_minutes = (time.perf_counter() - total_start) / 60
print()
print(f"Total runtime: {total_minutes:.1f} minutes")

## 10. Xem kết quả

In [ ]:
results = cache.load_results(
    tasks=tasks,
    include_remote=False,
)

df = results.to_dataframe(format="long")
display(df)

## 11. Điểm trung bình RAG Core

Đây không phải official VN-MTEB overall score.  
Đây là trung bình main score của các Retrieval task đã chọn.

In [ ]:
avg_score = df["score"].mean()
print(f"RAG Core average NDCG@10: {avg_score:.4f}")

## 12. Xuất CSV

In [ ]:
CSV_PATH = str(RUNTIME_DIR / "VietEmbed-RAG-V1_RAG-Core.csv")

df.to_csv(CSV_PATH, index=False)

print("Saved:", CSV_PATH)

## 13. Đóng gói cache

In [ ]:
import shutil

ZIP_PATH = str(RUNTIME_DIR / "VietEmbed-RAG-V1_RAG-Core-results")

shutil.make_archive(
    ZIP_PATH,
    "zip",
    RESULT_DIR,
)

print("Saved:", ZIP_PATH + ".zip")

## 14. Download kết quả

In [ ]:
try:
    from google.colab import files
except ImportError:
    print("Không chạy trong Colab. Kết quả nằm tại:")
    print("-", CSV_PATH)
    print("-", ZIP_PATH + ".zip")
else:
    files.download(CSV_PATH)
    files.download(ZIP_PATH + ".zip")

# Runtime dự kiến

Với E5-base, `BATCH_SIZE = 32`:

### T4 16 GB
- Không Quora: khoảng 20–60 phút
- Có Quora: khoảng 45–120 phút

### L4 24 GB
- Không Quora: khoảng 10–30 phút
- Có Quora: khoảng 20–60 phút

Nếu đánh giá V1 nghiêm túc, giữ:

```python
INCLUDE_QUORA = True
```

Nếu chỉ smoke test sau fine-tune:

```python
INCLUDE_QUORA = False
```

# Khi nào chạy full VN-MTEB Retrieval?

Chỉ nên chạy full khi model gần final hoặc chuẩn bị publish/model card, vì các task như
MSMARCO-VN, HotpotQA-VN, FEVER-VN, DBPedia-VN và NQ-VN có corpus hàng triệu documents.